In [1]:
# pip install pymupdf pandas tqdm openai  (or anthropic)

import fitz                      # PyMuPDF
import re
import json
import csv
import hashlib
from pathlib import Path
from datetime import datetime
import pandas as pd
from tqdm import tqdm

# Paths
PDF_DIR     = Path("gen_data/IFRS")
OUT_DIR     = Path("gen_data/IFRS/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

PARA_JSON   = OUT_DIR / "ifrs_paragraphs.json"
CATALOG_CSV = OUT_DIR / "disclosures_source_grounded.csv"
CATALOG_JSON= OUT_DIR / "disclosures_source_grounded.json"
VAL_REPORT  = OUT_DIR / "validation_report.md"

PDF_FILES = {
    "IFRS_S1": PDF_DIR / "ifrs_s1.pdf",
    "IFRS_S2": PDF_DIR / "ifrs_s2.pdf",
}

In [2]:
def extract_blocks(pdf_path: Path) -> list[dict]:
    """
    Extract all text blocks from a PDF with layout metadata.
    Returns list of dicts: {page, block_no, text, font_size, is_bold, bbox}
    """
    doc = fitz.open(str(pdf_path))
    blocks = []

    for page_num, page in enumerate(doc, start=1):
        page_dict = page.get_text("dict", flags=fitz.TEXT_PRESERVE_WHITESPACE)

        for block in page_dict["blocks"]:
            if block["type"] != 0:          # 0 = text, 1 = image
                continue

            block_text_parts = []
            font_sizes = []
            bold_flags = []

            for line in block["lines"]:
                for span in line["spans"]:
                    text = span["text"].strip()
                    if text:
                        block_text_parts.append(text)
                        font_sizes.append(span["size"])
                        bold_flags.append(bool(span["flags"] & 2**4))  # bold bit

            full_text = " ".join(block_text_parts).strip()
            if not full_text:
                continue

            blocks.append({
                "page":       page_num,
                "block_no":   block["number"],
                "text":       full_text,
                "font_size":  round(sum(font_sizes) / len(font_sizes), 2) if font_sizes else 0,
                "is_bold":    any(bold_flags),
                "bbox":       block["bbox"],   # (x0, y0, x1, y1)
                "x0":         round(block["bbox"][0], 1),
            })

    doc.close()
    print(f"  {pdf_path.name}: {page_num} pages, {len(blocks)} blocks extracted")
    return blocks


raw_blocks = {}
for standard, pdf_path in PDF_FILES.items():
    print(f"Extracting: {standard}")
    raw_blocks[standard] = extract_blocks(pdf_path)

Extracting: IFRS_S1
  ifrs_s1.pdf: 48 pages, 813 blocks extracted
Extracting: IFRS_S2
  ifrs_s2.pdf: 48 pages, 753 blocks extracted


In [3]:
# Paragraph ID pattern — covers all forms found in IFRS S1/S2:
#   27  6  44  B62  B62A  D4  E1
#   6(a)  25(a)  29(a)(i)  29(a)(i)(1)
PARA_ID_PATTERN = re.compile(
    r"""
    ^                           # must be at start of text
    (?P<para_id>
        (?:[A-Z]\d+[A-Z]?)      # appendix: B62, B62A, D4, E1
        |
        (?:\d{1,3}              # base number 1–999
            (?:\([a-z]{1,3}\)   # (a), (b), (ab)
                (?:\([ivxlc]+\) # (i), (ii), (iv)
                    (?:\(\d+\)) # (1), (2)
                ?)?
            )?
        )
    )
    (?:\s{2,}|\t)               # followed by 2+ spaces or tab (not just one)
    (?P<rest>.+)                # the paragraph text
    """,
    re.VERBOSE | re.DOTALL
)

# Section headings we use to assign topic
SECTION_HEADING_MAP = {
    r"governance":          "Governance",
    r"strategy":            "Strategy",
    r"risk\s+management":   "Risk_management",
    r"metrics\s+and\s+targets": "Metrics_targets",
}

# Core content page ranges per standard — verified against PDF structure
# Adjust these after you open the PDFs and check page numbers
CORE_CONTENT_PAGES = {
    "IFRS_S1": (7, 22),    # pages 7–22 contain paragraphs 25–53
    "IFRS_S2": (6, 20),    # pages 6–20 contain paragraphs 5–37
}


def parse_paragraphs(blocks: list[dict], standard: str) -> dict[str, dict]:
    """
    Parse paragraph IDs and text from raw blocks.
    
    Returns dict: paragraph_id → {
        id, standard, text, page, topic, is_core_content,
        raw_blocks (list of block texts that were merged)
    }
    """
    start_page, end_page = CORE_CONTENT_PAGES[standard]
    
    paragraphs = {}
    current_id  = None
    current_text_parts = []
    current_page = None
    current_topic = "Unknown"

    def flush_current():
        if current_id is None:
            return
        full_text = " ".join(current_text_parts)
        full_text = re.sub(r"\s{2,}", " ", full_text).strip()
        is_core = start_page <= current_page <= end_page
        paragraphs[current_id] = {
            "id":              current_id,
            "standard":        standard,
            "topic":           current_topic,
            "text":            full_text,
            "page":            current_page,
            "is_core_content": is_core,
            "char_count":      len(full_text),
        }

    for block in blocks:
        text = block["text"]

        # Detect section heading — update current_topic
        for pattern, topic_name in SECTION_HEADING_MAP.items():
            if re.match(pattern, text.strip(), re.IGNORECASE) and block["is_bold"]:
                current_topic = topic_name
                break

        # Try to match a paragraph ID at the start of this block
        match = PARA_ID_PATTERN.match(text)
        if match:
            flush_current()                        # save previous paragraph
            current_id         = match.group("para_id")
            current_text_parts = [match.group("rest").strip()]
            current_page       = block["page"]
        elif current_id is not None:
            # Continuation block — append to current paragraph
            current_text_parts.append(text)

    flush_current()   # save last paragraph

    core_count = sum(1 for p in paragraphs.values() if p["is_core_content"])
    print(f"  {standard}: {len(paragraphs)} total paragraphs, "
          f"{core_count} in core content section")
    return paragraphs


all_paragraphs = {}
for standard, blocks in raw_blocks.items():
    print(f"\nParsing: {standard}")
    all_paragraphs[standard] = parse_paragraphs(blocks, standard)

# Save ifrs_paragraphs.json — this file is never modified after this point
combined_paragraphs = {
    std: list(paras.values())
    for std, paras in all_paragraphs.items()
}
PARA_JSON.write_text(
    json.dumps(combined_paragraphs, indent=2, ensure_ascii=False),
    encoding="utf-8"
)
print(f"\nifrs_paragraphs.json saved → {PARA_JSON}")


Parsing: IFRS_S1
  IFRS_S1: 0 total paragraphs, 0 in core content section

Parsing: IFRS_S2
  IFRS_S2: 0 total paragraphs, 0 in core content section

ifrs_paragraphs.json saved → gen_data\IFRS\processed\ifrs_paragraphs.json
